In [0]:
%sql
-- DIM_Asteroide
CREATE TABLE IF NOT EXISTS nasa_dw.gold.dim_asteroide
USING DELTA
COMMENT 'Dimensión de asteroides'
AS SELECT
    asteroide_id, nombre, diametro_promedio_km, categoria_tamano,
    es_potencialmente_peligroso, magnitud_absoluta, url_nasa,
    fecha_acercamiento AS fecha_primera_deteccion,
    CAST(NULL AS INT) AS clasificacion_id,
    CURRENT_TIMESTAMP() AS fecha_actualizacion
FROM nasa_dw.silver.neo_clean WHERE 1=0;

MERGE INTO nasa_dw.gold.dim_asteroide AS tgt
USING (
    SELECT
        asteroide_id, nombre, diametro_promedio_km, categoria_tamano,
        es_potencialmente_peligroso, magnitud_absoluta, url_nasa,
        MIN(fecha_acercamiento) AS fecha_primera_deteccion,
        CAST(NULL AS INT) AS clasificacion_id,
        CURRENT_TIMESTAMP()     AS fecha_actualizacion
    FROM nasa_dw.silver.neo_clean
    GROUP BY asteroide_id, nombre, diametro_promedio_km, categoria_tamano,
             es_potencialmente_peligroso, magnitud_absoluta, url_nasa
) AS src
ON tgt.asteroide_id = src.asteroide_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;